In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

In [6]:
homes=pd.read_csv('RealEstate_Prices.csv')

In [8]:
homes.head()

,No,X1 transaction date,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,X5 latitude,X6 longitude,Y house price of unit area
0,1,2012.916667,32.0,84.87882,10,24.98298,121.54024,37.9
1,2,2012.916667,19.5,306.59470,9,24.98034,121.53951,42.2
2,3,2013.583333,13.3,561.98450,5,24.98746,121.54391,47.3
3,4,2013.500000,13.3,561.98450,5,24.98746,121.54391,54.8
4,5,2012.833333,5.0,390.56840,5,24.97937,121.54245,43.1


In [10]:
homes.columns=homes.columns.str.strip().str.lower().str.replace(r'[^a-z0-9]+','_',regex=True)
homes = homes.rename(columns={'no': 'id', 'x1_transaction_date': 'transaction_date',
    'x2_house_age': 'house_age', 'x3_distance_to_the_nearest_mrt_station': 'distance_to_mrt',
    'x4_number_of_convenience_stores': 'stores', 'x5_latitude': 'latitude',
    'x6_longitude': 'longitude', 'y_house_price_of_unit_area': 'price_per_unit'})

In [12]:
homes.head()

,id,transaction_date,house_age,distance_to_mrt,stores,latitude,longitude,price_per_unit
0,1,2012.916667,32.0,84.87882,10,24.98298,121.54024,37.9
1,2,2012.916667,19.5,306.59470,9,24.98034,121.53951,42.2
2,3,2013.583333,13.3,561.98450,5,24.98746,121.54391,47.3
3,4,2013.500000,13.3,561.98450,5,24.98746,121.54391,54.8
4,5,2012.833333,5.0,390.56840,5,24.97937,121.54245,43.1


In [14]:
homes.isna().sum()

id                  0
transaction_date    0
house_age           0
distance_to_mrt     0
stores              0
latitude            0
longitude           0
price_per_unit      0
dtype: int64

In [18]:
homes.duplicated().sum()

0

In [20]:
len(homes)

414

In [22]:
homes=homes.drop_duplicates().dropna(subset='price_per_unit')

In [32]:
features = homes.columns.difference(['id', 'price_per_unit'])
homes[features]=homes[features].fillna(homes[features].median())
print("Cleaned records:" ,len(homes))

Cleaned records: 414


In [34]:
selected=homes[homes['stores']>=5].copy()
selected.head()

,id,transaction_date,house_age,distance_to_mrt,stores,latitude,longitude,price_per_unit
0,1,2012.916667,32.0,84.87882,10,24.98298,121.54024,37.9
1,2,2012.916667,19.5,306.59470,9,24.98034,121.53951,42.2
2,3,2013.583333,13.3,561.98450,5,24.98746,121.54391,47.3
3,4,2013.500000,13.3,561.98450,5,24.98746,121.54391,54.8
4,5,2012.833333,5.0,390.56840,5,24.97937,121.54245,43.1


In [36]:
len(selected)

200

In [38]:
selected['age_group']=pd.cut(selected['house_age'],bins=[-1,10,30,100],
                             labels=['up to 10' , '10 to 30' , 'over 30'])
summ=selected.groupby(['age_group','stores'],observed=True)['price_per_unit'].agg(['count','mean','median'])
display(summ)

count       mean  median
age_group stores                          
up to 10  5          22  50.840909   51.75
          6          18  54.316667   49.85
          7           8  51.312500   51.60
          8           3  52.600000   56.80
          9          16  53.987500   54.90
          10          3  55.366667   54.40
10 to 30  5          40  42.595000   42.20
          6          11  39.790909   40.80
          7          13  42.507692   42.00
          8          10  41.210000   41.25
          9           2  42.100000   42.10
          10          2  45.300000   45.30
over 30   5           5  34.920000   34.70
          6           8  40.225000   36.50
          7          10  39.620000   40.45
          8          17  45.352941   42.30
          9           7  49.328571   39.70
          10          5  45.520000   47.10

In [42]:
q1=selected['price_per_unit'].quantile(0.25)
q3=selected['price_per_unit'].quantile(0.75)
iqr=q3-q1
upper=q3+1.5*iqr
lower=q1-1.5*iqr
outliers=~selected['price_per_unit'].between(lower,upper)
print(outliers.sum())

7


In [44]:
#OHE
encoded=pd.get_dummies(selected,columns=['age_group'],dtype=int)
Path('outputs').mkdir(exist_ok=True)
encoded.to_csv('output.csv',index=False)
display(encoded.head())

,id,transaction_date,house_age,distance_to_mrt,stores,latitude,longitude,price_per_unit,age_group_up to 10,age_group_10 to 30,age_group_over 30
0,1,2012.916667,32.0,84.87882,10,24.98298,121.54024,37.9,0,0,1
1,2,2012.916667,19.5,306.59470,9,24.98034,121.53951,42.2,0,1,0
2,3,2013.583333,13.3,561.98450,5,24.98746,121.54391,47.3,0,1,0
3,4,2013.500000,13.3,561.98450,5,24.98746,121.54391,54.8,0,1,0
4,5,2012.833333,5.0,390.56840,5,24.97937,121.54245,43.1,1,0,0
